# 02 · Data preparation

Builds the tables every later notebook uses: one row per patient (with its cross-validation fold) and one row per box. It also decides how images are stored and checks the output of `scripts/prepare_data.py`.

Needs the Kaggle data in `../data/raw` (see `configs/paths.yaml`), the DICOM header cache written by 01_eda, and a run of `python scripts/prepare_data.py` before section 5.

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

DATA_DIR = Path("../../data/raw/rsna-pneumonia-detection-challenge").resolve()
OUT_DIR = DATA_DIR.parents[1] / "processed"

labels = pd.read_csv(DATA_DIR / "stage_2_train_labels.csv")
classes = pd.read_csv(DATA_DIR / "stage_2_detailed_class_info.csv")
meta = pd.read_csv(OUT_DIR / "dicom_meta.csv")  # DICOM headers, cached in 01_eda

## 1. One row per patient

**Question:** Every later notebook needs the same patient list with its label and metadata. The CSVs have one row per box, so we build a clean table once: one row per patient (= one image).

In [2]:
n_boxes = labels.groupby("patientId")["x"].count().rename("n_boxes")

patients = (
    labels.groupby("patientId")["Target"].max().to_frame()
    .join(classes.drop_duplicates("patientId").set_index("patientId"))
    .join(n_boxes)
    .join(meta[meta["split"] == "train"].set_index("patientId")[["ViewPosition", "PatientSex", "PatientAge", "PixelSpacing"]])
    .reset_index()
)
patients["PixelSpacing"] = patients["PixelSpacing"].str.extract(r"\[([\d.]+)")[0].astype(float).round(3)
patients.columns = ["patient_id", "target", "class", "n_boxes", "view", "sex", "age", "pixel_spacing"]
patients.head()

,patient_id,target,class,n_boxes,view,sex,age,pixel_spacing
0,0004cfab-14fd-4e49-80ba-63a80b6bddd6,0,No Lung Opacity / Not Normal,0,PA,F,51,0.143
1,000924cf-0f8d-42bd-9158-1af53881a557,0,Normal,0,AP,F,19,0.139
2,000db696-cf54-4385-b10b-6b16fbb3f985,1,Lung Opacity,2,AP,F,25,0.168
3,000fe35a-2649-43d4-b027-e67796d412e0,1,Lung Opacity,2,AP,M,40,0.171
4,001031d9-f904-4a23-b3e5-2c088acd19c6,1,Lung Opacity,2,PA,M,57,0.139


Sanity checks against the numbers from 01_eda:

In [3]:
print("patients           :", len(patients), "| unique:", patients["patient_id"].is_unique)
print("missing values     :", patients.isna().sum().sum())
print("boxes              :", patients["n_boxes"].sum())
print("target == has box  :", (patients["target"] == (patients["n_boxes"] > 0)).all())
patients["class"].value_counts()

patients           : 26684 | unique: True
missing values     : 0
boxes              : 9555
target == has box  : True


class
No Lung Opacity / Not Normal    11821
Normal                           8851
Lung Opacity                     6012
Name: count, dtype: int64

## 2. One row per box

**Question:** The detection models need the boxes. We keep only the rows that have a box and check them: are the coordinates whole numbers (so converting to int loses nothing), is every box non-empty, and does every box lie inside the 1024 x 1024 image?

In [4]:
cols = ["x", "y", "width", "height"]
boxes = labels.dropna(subset=["x"]).drop(columns="Target").rename(columns={"patientId": "patient_id"})
print("boxes                   :", len(boxes))
print("missing coordinates     :", boxes[cols].isna().sum().sum())
print("non-integer coordinates :", (boxes[cols] % 1 != 0).sum().sum())
boxes[cols] = boxes[cols].astype(int)

inside = (boxes["x"] >= 0) & (boxes["y"] >= 0) & (boxes["x"] + boxes["width"] <= 1024) & (boxes["y"] + boxes["height"] <= 1024)
print("zero width or height    :", ((boxes["width"] <= 0) | (boxes["height"] <= 0)).sum())
print("outside the image       :", (~inside).sum())
print("duplicate boxes         :", boxes.duplicated().sum())
boxes.head()

boxes                   : 9555
missing coordinates     : 0
non-integer coordinates : 0
zero width or height    : 0
outside the image       : 0
duplicate boxes         : 0


,patient_id,x,y,width,height
4,00436515-870c-4b36-a041-de91049b9ab4,264,152,213,379
5,00436515-870c-4b36-a041-de91049b9ab4,562,152,256,453
8,00704310-78a8-4b38-8475-49f4573b2dbb,323,577,160,104
9,00704310-78a8-4b38-8475-49f4573b2dbb,695,575,162,137
14,00aecb01-a116-45a2-956c-08d2fa55433f,288,322,94,135


## 3. Cross-validation folds

**Question:** How do we split the training data so that every fold is a fair test? We use 5 folds: a model trains on 4 and is scored on the 5th. Each patient has one image, so no patient can end up in two folds.

We stratify on **class × view**: 01_eda showed that both the class and the view (AP 38% vs PA 9% Lung Opacity) change the difficulty, so every fold should have the same mix of both.

In [5]:
from sklearn.model_selection import StratifiedKFold

N_FOLDS, SEED = 5, 42
strata = patients["class"] + " | " + patients["view"]
print(strata.value_counts(), "\n")

patients = patients.reset_index(drop=True)  # StratifiedKFold returns positions; with a 0..n-1 index they equal the labels
patients["fold"] = -1
for fold, (_, val_idx) in enumerate(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(patients, strata)):
    patients.loc[val_idx, "fold"] = fold
patients["fold"].value_counts().sort_index()

Normal | PA                          7214
No Lung Opacity / Not Normal | PA    5949
No Lung Opacity / Not Normal | AP    5872
Lung Opacity | AP                    4664
Normal | AP                          1637
Lung Opacity | PA                    1348
Name: count, dtype: int64 



fold
0    5337
1    5337
2    5337
3    5337
4    5336
Name: count, dtype: int64

Is every fold alike? Class and view shares (%), mean number of boxes per Lung Opacity patient, and mean age:

In [6]:
pos = patients[patients["target"] == 1]
(pd.crosstab(patients["fold"], patients["class"], normalize="index") * 100).round(1).assign(
    pct_AP=(patients.groupby("fold")["view"].apply(lambda s: (s == "AP").mean()) * 100).round(1),
    boxes_per_pos=pos.groupby("fold")["n_boxes"].mean().round(2),
    mean_age=patients.groupby("fold")["age"].mean().round(1),
)

class,Lung Opacity,No Lung Opacity / Not Normal,Normal,pct_AP,boxes_per_pos,mean_age
fold,,,,,,
0,22.5,44.3,33.2,45.6,1.59,47.0
1,22.5,44.3,33.2,45.6,1.58,47.1
2,22.5,44.3,33.1,45.6,1.62,47.1
3,22.5,44.3,33.2,45.6,1.58,47.1
4,22.5,44.3,33.2,45.6,1.58,46.9


### 3.1 Key findings

- **`patients` has one row per patient:** 26,684 patients with target, class, number of boxes, view, sex, age and pixel spacing. No values are missing and the counts match 01_eda.
- **`boxes` has one row per box:** 9,555 boxes. All coordinates are whole numbers, no box is empty, all lie inside the image and none is duplicated, so no cleaning is needed. The same checks run as assertions in `scripts/prepare_data.py`, so the script stops if the labels ever break them.
- **The 5 folds are alike.** Each has 5,336–5,337 patients with the same class shares (22.5 / 44.3 / 33.2%) and the same AP share (45.6%). Boxes per Lung Opacity patient (1.58–1.62) and mean age (~47) also match, although we did not stratify on them. So the folds do not differ in these basic properties. Scores can still differ between folds, because box size, image quality and training randomness were not controlled.

## 4. Which format should we read images from?

**Question:** Many solutions converted the DICOMs to PNG first, assuming DICOM is slow to read. Is that true here? We save the same 1,000 random images in each format to a temporary folder, then time how long one image takes to read and how big the file is. The files were just written or read, so they sit in the operating system's memory cache: these are **warm-cache** read times, not first reads from disk.

In [7]:
import shutil
import time

import cv2
import numpy as np
import pydicom

TRAIN_IMG = DATA_DIR / "stage_2_train_images"
BENCH_DIR = OUT_DIR / "format_benchmark"
BENCH_DIR.mkdir(parents=True, exist_ok=True)
ids = patients["patient_id"].sample(1000, random_state=1).tolist()

identical = True
for pid in ids:
    img = pydicom.dcmread(TRAIN_IMG / f"{pid}.dcm").pixel_array
    cv2.imwrite(str(BENCH_DIR / f"{pid}.png"), img)
    np.save(BENCH_DIR / f"{pid}.npy", img)
    cv2.imwrite(str(BENCH_DIR / f"{pid}_512.png"), cv2.resize(img, (512, 512), interpolation=cv2.INTER_AREA))
    identical &= np.array_equal(img, cv2.imread(str(BENCH_DIR / f"{pid}.png"), cv2.IMREAD_UNCHANGED))
print("PNG 1024 identical to DICOM pixels:", identical, "| dtype:", img.dtype, "| shape:", img.shape)

PNG 1024 identical to DICOM pixels: True | dtype: uint8 | shape: (1024, 1024)


In [8]:
formats = {
    "dicom": (lambda p: pydicom.dcmread(TRAIN_IMG / f"{p}.dcm").pixel_array, lambda p: TRAIN_IMG / f"{p}.dcm"),
    "png_1024": (lambda p: cv2.imread(str(BENCH_DIR / f"{p}.png"), cv2.IMREAD_UNCHANGED), lambda p: BENCH_DIR / f"{p}.png"),
    "npy_1024": (lambda p: np.load(BENCH_DIR / f"{p}.npy"), lambda p: BENCH_DIR / f"{p}.npy"),
    "png_512": (lambda p: cv2.imread(str(BENCH_DIR / f"{p}_512.png"), cv2.IMREAD_UNCHANGED), lambda p: BENCH_DIR / f"{p}_512.png"),
}

def ms_per_image(read):
    t = time.perf_counter()
    for pid in ids:
        read(pid)
    return (time.perf_counter() - t) / len(ids) * 1e3

n_images = len(patients) + 3000
rows = []
for name, (read, path) in formats.items():
    kb = np.mean([path(p).stat().st_size for p in ids]) / 1e3
    rows.append({"format": name, "read_ms": ms_per_image(read),
                 "kb_per_image": kb, "total_gb": kb * n_images / 1e6})
shutil.rmtree(BENCH_DIR)

pd.DataFrame(rows).set_index("format").round(1)

,read_ms,kb_per_image,total_gb
format,,,
dicom,3.3,134.6,4.0
png_1024,6.8,400.6,11.9
npy_1024,0.5,1048.7,31.1
png_512,2.1,116.0,3.4


`dicom` is the original files; like the others, its total is estimated from the 1,000-image sample. `png_512` is included to show what a smaller cache would give, but it would fix the input size before we have tested 256 / 384 / 512.

### 4.1 Key findings

- **Converting to PNG makes things worse here.** The DICOMs hold JPEG-compressed pixels, and JPEG decodes fast. A PNG of the same image is about 3x larger and about 2x slower to read, with no gain in quality (both are the same 8-bit pixels).
- **These are warm-cache times.** A first read from disk is slower, and more so for NPY, which reads ~8x more bytes per image (1 MB vs 135 KB). With 64 GB of RAM, most of the 31 GB NPY cache can stay in memory after the first epoch, so the gap should hold during training; we will confirm it with the real data loader.
- **Reading the DICOMs directly would also work.** One image takes 3–4 ms (timings vary a little between runs), and it needs no extra disk.
- **We choose NPY 1024.** It reads 5–7x faster than DICOM from the cache (depending on the run) and keeps the exact pixels at full resolution, so every input size (256 / 384 / 512) stays open. It costs ~31 GB, but for this project disk space is plentiful and time matters more, so faster loading is the better trade-off.

## 5. Running the full preparation

**Question:** The full conversion runs outside this notebook, in `scripts/prepare_data.py`, on all CPU cores (about half a minute on 16 cores). It uses the same steps as sections 1–3, moved into `src/rsna/data/prepare.py`, and writes the NPY files plus `patients.csv`, `boxes.csv` and `test.csv`. Did it produce exactly what we built above?

```bash
python scripts/prepare_data.py
```

In [9]:
saved_patients = pd.read_csv(OUT_DIR / "patients.csv")
saved_boxes = pd.read_csv(OUT_DIR / "boxes.csv")
saved_test = pd.read_csv(OUT_DIR / "test.csv")

print("patients.csv == patients above :", saved_patients.equals(patients))
print("boxes.csv    == boxes above    :", saved_boxes.equals(boxes.reset_index(drop=True)))
print("test.csv rows                  :", len(saved_test))
saved_test.head()

patients.csv == patients above : True
boxes.csv    == boxes above    : True
test.csv rows                  : 3000


,patient_id,view,sex,age,pixel_spacing
0,0000a175-0e68-4ca4-b1af-167204a7e0bc,PA,F,46,0.194
1,0005d3cc-3c3f-40b9-93c3-46231c3eb813,PA,F,22,0.143
2,000686d7-f4fc-448d-97a0-44fa9c5d3aa6,PA,M,64,0.143
3,000e3a7d-c0ca-4349-bb26-5af2d8993c3d,PA,F,75,0.143
4,00100a24-854d-423d-a092-edcf6179e061,AP,F,66,0.139


Is there one NPY per DICOM, and is every file complete? A 1024 x 1024 uint8 array saved with `np.save` is always 1,048,704 bytes (128-byte header + 1,048,576 pixels), so we check the size of **every** file. Comparing pixels with the DICOM is slower, so we do that for 500 random images per split:

In [10]:
rows = []
for split, ids_all in [("train", saved_patients["patient_id"]), ("test", saved_test["patient_id"])]:
    npy_dir = OUT_DIR / "npy_1024" / split
    npy_files = list(npy_dir.glob("*.npy"))
    sample = ids_all.sample(500, random_state=0)
    same = all(np.array_equal(np.load(npy_dir / f"{pid}.npy"),
                              pydicom.dcmread(DATA_DIR / f"stage_2_{split}_images" / f"{pid}.dcm").pixel_array) for pid in sample)
    rows.append({"split": split, "n_dicom": len(list((DATA_DIR / f"stage_2_{split}_images").glob("*.dcm"))),
                 "n_npy": len(npy_files), "ids_match": {f.stem for f in npy_files} == set(ids_all),
                 "all_complete": all(f.stat().st_size == 1_048_704 for f in npy_files), "sample_identical": same, "size_gb": sum(f.stat().st_size for f in npy_files) / 1e9})
pd.DataFrame(rows).set_index("split").round(1)

,n_dicom,n_npy,ids_match,all_complete,sample_identical,size_gb
split,,,,,,
train,26684,26684,True,True,True,28.0
test,3000,3000,True,True,True,3.1


### 5.1 Key findings

- **The script reproduces this notebook exactly:** `patients.csv` and `boxes.csv` are equal to the tables built above.
- **Every DICOM has a complete NPY:** 26,684 train and 3,000 test files, ~31 GB in total, all with the expected size. The 500 checked images per split have exactly the same pixels as their DICOMs.
- From now on, notebooks read images from `npy_1024/` and labels from `patients.csv` (with the fold) and `boxes.csv`. `test.csv` holds the test ids with their view, sex, age and pixel spacing.